# Data-gen pipeline: SMILES → torch_geometric Data

This notebook builds a full `torch_geometric.data.Data` object using the pipeline.
Provide DeepMD models (pot/dipole/polar) or use Psi4 for small molecules.


In [1]:
from pathlib import Path
import sys
import torch

root = Path.cwd()
if (root / "data-gen-pipeline").is_dir():
    pipe_dir = (root / "data-gen-pipeline").resolve()
elif root.name == "data-gen-pipeline":
    pipe_dir = root.resolve()
else:
    raise RuntimeError("Run this notebook from the repo root or data-gen-pipeline/")

sys.path.insert(0, str(pipe_dir))

import pipeline
from deepmd_backend import DeepMDDipoleBackend, DeepMDPolarBackend, DeepMDPotHessianBackend
from psi4_backend import Psi4HessianBackend


In [2]:
smiles = "C"
pot_model = pipe_dir / "checkpoints/dpa2_multi_Domains_Drug.pth"

# Set these if you have DeepMD dipole/polar models.
dipole_model = None
polar_model = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = pipeline.PipelineConfig(
    output_dir=Path("data/gen"),
    device=device,
    dft_atom_cutoff=20,
    graph_k=20,
    graph_clamp_min=0.5,
    graph_clamp_max=100.0,
    pos_step=1e-3,
    allow_missing_polar=True,
    allow_missing_dipole=True,
    psi4_fallback=True,
)


In [3]:
dipole_backend = (
    DeepMDDipoleBackend(device=device, model_path=str(dipole_model))
    if dipole_model
    else None
)
polar_backend = (
    DeepMDPolarBackend(device=device, model_path=str(polar_model))
    if polar_model
    else None
)

hessian_backend = DeepMDPotHessianBackend(
    device=device,
    model_path=str(pot_model),
)

psi4_backend = None
try:
    psi4_backend = Psi4HessianBackend(
        device=torch.device("cpu"),
        method=cfg.psi4_method,
        basis=cfg.psi4_basis,
        charge=cfg.psi4_charge,
        multiplicity=cfg.psi4_multiplicity,
        num_threads=cfg.psi4_threads,
        memory=cfg.psi4_memory,
        scf_type=cfg.psi4_scf_type,
        guess=cfg.psi4_guess,
        quiet=cfg.psi4_quiet,
    )
except Exception as exc:
    print("Psi4 not available:", exc)

if dipole_backend is None and psi4_backend is None and not cfg.allow_missing_dipole:
    raise RuntimeError("Provide a DeepMD dipole model or install Psi4 for small molecules, or set allow_missing_dipole=True.")


To get the best performance, it is recommended to adjust the number of threads by setting the environment variables OMP_NUM_THREADS, DP_INTRA_OP_PARALLELISM_THREADS, and DP_INTER_OP_PARALLELISM_THREADS. See https://deepmd.rtfd.io/parallelism/ for more information.
You can use the environment variable DP_INFER_BATCH_SIZE tocontrol the inference batch size (nframes * natoms). The default value is 1024.


Psi4 not available: psi4 is required for Psi4HessianBackend


In [4]:
item = pipeline.SmilesItem(number=1, smile=smiles)
data = pipeline.build_data_from_smiles(
    item,
    cfg,
    dipole_backend,
    polar_backend,
    hessian_backend,
    psi4_backend,
)
data


Data(edge_index=[2, 20], pos=[5, 3], z=[5], number=1, smile='C', energy=[1, 1], dipole=[1, 3], npacharge=[5], polar=[1, 3, 3], quadrupole=[1, 3, 3], octapole=[1, 3, 3, 3], hyperpolar=[1, 3, 3, 3], Hi=[5, 3, 3], Hij=[20, 3, 3], dedipole=[5, 3, 3], depolar=[5, 3, 6], tran_dipole=[1, 10, 3], tran_energy=[1, 10])

#### should see: 
##### Data(edge_index=[2, 20], pos=[5, 3], number=1, smile='C', z=[5], quadrupole=[1, 3, 3], octapole=[1, 3, 3, 3], npacharge=[5], dipole=[1, 3], polar=[1, 3, 3], hyperpolar=[1, 3, 3, 3], energy=[1, 1], Hij=[20, 3, 3], Hi=[5, 3, 3], dedipole=[5, 3, 3], depolar=[5, 3, 6], tran_dipole=[1, 10, 3], tran_energy=[1, 10])


In [5]:
def shape(val):
    try:
        return tuple(val.shape)
    except Exception:
        return None

for key in data.keys():
    print(f"{key}: {shape(getattr(data, key))}")


z: (5,)
npacharge: (5,)
pos: (5, 3)
smile: None
dipole: (1, 3)
quadrupole: (1, 3, 3)
depolar: (5, 3, 6)
number: None
hyperpolar: (1, 3, 3, 3)
tran_dipole: (1, 10, 3)
edge_index: (2, 20)
polar: (1, 3, 3)
Hij: (20, 3, 3)
tran_energy: (1, 10)
Hi: (5, 3, 3)
dedipole: (5, 3, 3)
octapole: (1, 3, 3, 3)
energy: (1, 1)
